# Windowed Minority Guidance â€” Extended Sweep (v2)

Sweeps `guidance_scale` in `[0.5, 2.0, 3.5, 7.0]` across all 5 conditions and 50 seeds.
Includes boundary-sensitivity probe at `scale=3.5` with `boundary_set` in
`[equal_thirds, splits_250_750, splits_300_700]`.

**Do not modify the Parameters cell manually** â€” `wmg-kaggle-runner` injects values there.

Output CSV: `/kaggle/working/runs_{GUIDANCE_SCALE}_{DATASET}.csv`

Schema (all columns present even if empty):
`run_id, condition, guidance_window, t_start, t_end, guidance_scale, dataset,
boundary_set, seed, n_generated, classifier_mean_confidence, classifier_mean_loss,
runtime_seconds, artifact_path, timestamp`

In [ ]:
# -- PARAMETERS (injected by wmg-kaggle-runner) ------
GUIDANCE_SCALE = 7.0            # float
DATASET        = "lsun_bedroom" # str
BOUNDARY_SET   = "equal_thirds" # str
N_SEEDS        = 50             # int
GAP_PAIRS      = None           # None for non-continuation
RUN_MODE       = "sweep"        # str


In [ ]:
# â”€â”€ CELL 1: Install dependencies + clone repo â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import subprocess, sys, os

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "blobfile", "scikit-learn", "lpips", "mpi4py"],
    check=True
)

REPO = "/kaggle/working/minority-guidance"
if not os.path.exists(REPO):
    subprocess.run(
        ["git", "clone", "-q",
         "https://github.com/soobin-um/minority-guidance", REPO],
        check=True
    )
print("Repo ready.")

In [ ]:
# â”€â”€ CELL 2: Patch unet.py â€” confirm or add f_out â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
#
# EncoderUNetModel.forward must accept f_out=False.
# When f_out=True the method returns the 8x8x512 bottleneck feature
# map h instead of the classifier head output. This is the mechanism
# used by cond_fn to extract latent features for the minority classifier.
#
# The patch is idempotent: if f_out is already present, it is a no-op.

unet_path = f"{REPO}/guided_diffusion/unet.py"

with open(unet_path, "r") as fh:
    unet_src = fh.read()

if "f_out" in unet_src:
    print("unet.py: f_out already present â€” CONFIRMED_PRESENT, no patch needed.")
    F_OUT_STATUS = "CONFIRMED_PRESENT"
else:
    # Patch the forward signature
    old_sig = "    def forward(self, x, timesteps):\n"
    new_sig = "    def forward(self, x, timesteps, f_out=False):\n"
    assert unet_src.count(old_sig) == 1, (
        f"Expected exactly 1 EncoderUNetModel.forward signature, "
        f"found {unet_src.count(old_sig)} â€” repo structure has drifted."
    )
    unet_src = unet_src.replace(old_sig, new_sig, 1)

    # Patch the return path: before `return self.out(h)` in the else branch
    old_ret = (
        "        else:\n"
        "            h = h.type(x.dtype)\n"
        "            return self.out(h)\n"
    )
    new_ret = (
        "        else:\n"
        "            h = h.type(x.dtype)\n"
        "            if f_out:\n"
        "                return h\n"
        "            return self.out(h)\n"
    )
    assert old_ret in unet_src, (
        "unet.py patch target (else-branch return) not found â€” repo may have changed."
    )
    unet_src = unet_src.replace(old_ret, new_ret, 1)

    with open(unet_path, "w") as fh:
        fh.write(unet_src)
    print("unet.py: f_out parameter ADDED successfully.")
    F_OUT_STATUS = "ADDED"

print(f"f_out status: {F_OUT_STATUS}")

In [ ]:
# â”€â”€ CELL 3: Patch script_util.py â€” add f_extractor_defaults â”€â”€

script_util_path = f"{REPO}/guided_diffusion/script_util.py"
with open(script_util_path, "r") as fh:
    content = fh.read()

if "f_extractor_defaults" not in content:
    with open(script_util_path, "a") as fh:
        fh.write('''

def f_extractor_defaults(use_fp16=False):
    """Defaults for the 256x256 ImageNet classifier used as feature extractor."""
    return dict(
        image_size=256,
        classifier_use_fp16=use_fp16,
        classifier_width=128,
        classifier_depth=2,
        classifier_attention_resolutions="32,16,8",
        classifier_use_scale_shift_norm=True,
        classifier_resblock_updown=True,
        classifier_pool="attention",
        in_channels=3,
        out_channels=1000
    )
''')
    print("script_util.py: appended f_extractor_defaults.")
else:
    print("script_util.py: f_extractor_defaults already present.")

In [ ]:
# â”€â”€ CELL 4: Locate and copy checkpoints â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import shutil, glob

ckpt_dir = f"{REPO}/ckpt"
os.makedirs(ckpt_dir, exist_ok=True)

# mc_lsun.pt â€” Kaggle dataset: vittorialanzo/mc-lsun
mc_candidates = glob.glob("/kaggle/input/**/mc_lsun.pt", recursive=True)
if mc_candidates:
    shutil.copy(mc_candidates[0], f"{ckpt_dir}/mc_lsun.pt")
    print(f"mc_lsun.pt copied from: {mc_candidates[0]}")
else:
    all_files = []
    for root, dirs, files in os.walk("/kaggle/input"):
        for fn in files:
            all_files.append(os.path.join(root, fn))
    print("=== ALL FILES UNDER /kaggle/input/ ===")
    for p in all_files:
        print(p)
    raise FileNotFoundError(
        "mc_lsun.pt not found under /kaggle/input/.\n"
        "Attach the Kaggle dataset vittorialanzo/mc-lsun to this notebook."
    )

# lsun_bedroom.pt (~2.1 GB) â€” OpenAI public storage
if not os.path.exists(f"{ckpt_dir}/lsun_bedroom.pt"):
    print("Downloading lsun_bedroom.pt (~2.1 GB)...")
    subprocess.run([
        "wget", "-q", "--show-progress",
        "-O", f"{ckpt_dir}/lsun_bedroom.pt",
        "https://openaipublic.blob.core.windows.net/diffusion/jul-2021/lsun_bedroom.pt"
    ], check=True)
    print("lsun_bedroom.pt done.")
else:
    print("lsun_bedroom.pt already present.")

# 256x256_classifier.pt (~206 MB) â€” OpenAI public storage
if not os.path.exists(f"{ckpt_dir}/256x256_classifier.pt"):
    print("Downloading 256x256_classifier.pt (~206 MB)...")
    subprocess.run([
        "wget", "-q", "--show-progress",
        "-O", f"{ckpt_dir}/256x256_classifier.pt",
        "https://openaipublic.blob.core.windows.net/diffusion/jul-2021/256x256_classifier.pt"
    ], check=True)
    print("256x256_classifier.pt done.")
else:
    print("256x256_classifier.pt already present.")

print("All checkpoints ready.")

In [ ]:
# â”€â”€ CELL 5: Imports and model loading â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import sys
if REPO not in sys.path:
    sys.path.insert(0, REPO)

import csv
import time
import uuid
import numpy as np
import torch as th
import torch.nn.functional as F
from datetime import datetime, timezone

from guided_diffusion import dist_util, logger
from guided_diffusion.script_util import (
    model_and_diffusion_defaults,
    classifier_defaults,
    create_model_and_diffusion,
    create_classifier,
    add_dict_to_argparser,
    args_to_dict,
    f_extractor_defaults,
)

dist_util.setup_dist()
logger.configure()

# â”€â”€ Model flags (LSUN Bedroom, verbatim from Um et al.) â”€â”€â”€â”€â”€â”€
import argparse

model_defaults = model_and_diffusion_defaults()
model_defaults.update({
    "attention_resolutions": "32,16,8",
    "class_cond": False,
    "diffusion_steps": 1000,
    "dropout": 0.1,
    "image_size": 256,
    "learn_sigma": True,
    "noise_schedule": "linear",
    "num_channels": 256,
    "num_head_channels": 64,
    "num_res_blocks": 2,
    "resblock_updown": True,
    "use_fp16": False,
    "use_scale_shift_norm": True,
    "timestep_respacing": "250",
    "rescale_timesteps": False,
})

args = argparse.Namespace(**model_defaults)
# Propagate GUIDANCE_SCALE parameter into args
args.classifier_scale = GUIDANCE_SCALE
args.clip_denoised = True
args.use_ddim = False
args.batch_size = 1

logger.log("Creating model and diffusion...")
model, diffusion = create_model_and_diffusion(
    **args_to_dict(args, model_and_diffusion_defaults().keys())
)
model.load_state_dict(
    dist_util.load_state_dict(f"{ckpt_dir}/lsun_bedroom.pt", map_location="cpu")
)
model.to(dist_util.dev())
model.eval()
logger.log("Model ready.")

# â”€â”€ Feature extractor (256x256 ImageNet classifier) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
f_ext_cfg = f_extractor_defaults()
f_extractor = create_classifier(**f_ext_cfg)
f_extractor.load_state_dict(
    dist_util.load_state_dict(f"{ckpt_dir}/256x256_classifier.pt", map_location="cpu")
)
f_extractor.to(dist_util.dev())
f_extractor.eval()
logger.log("Feature extractor ready.")

# â”€â”€ Minority classifier (mc_lsun.pt) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
clf_args = argparse.Namespace(**{
    "image_size": 8,        # latent_size
    "in_channels": 512,
    "out_channels": 100,
    "classifier_attention_resolutions": "8",
    "classifier_depth": 2,
    "classifier_width": 128,
    "classifier_pool": "attention",
    "classifier_resblock_updown": False,
    "classifier_use_scale_shift_norm": True,
    "classifier_use_fp16": False,
})
classifier = create_classifier(**args_to_dict(clf_args, classifier_defaults().keys()))
classifier.load_state_dict(
    dist_util.load_state_dict(f"{ckpt_dir}/mc_lsun.pt", map_location="cpu")
)
classifier.to(dist_util.dev())
classifier.eval()
logger.log("Minority classifier ready.")

MANUAL_CLASS_ID = 99  # LSUN Bedroom minority class
print(f"Models loaded. guidance_scale={GUIDANCE_SCALE}, dataset={DATASET}, boundary_set={BOUNDARY_SET}")

In [ ]:
# â”€â”€ CELL 6: Window boundary definitions â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
#
# t values are raw indices in [0, 999] (rescale_timesteps=False).
# Windows are half-open intervals [t_start, t_end).
#
# boundary_set options:
#   equal_thirds    : canonical Um et al. split â€” [0,333), [333,667), [667,1000)
#   splits_250_750  : boundary-sensitivity probe â€” [0,250), [250,750), [750,1000)
#   splits_300_700  : boundary-sensitivity probe â€” [0,300), [300,700), [700,1000)

BOUNDARY_SETS = {
    "equal_thirds": {
        "early_start": 0,
        "early_end":   333,
        "mid_start":   333,
        "mid_end":     667,
        "late_start":  667,
        "late_end":    1000,
    },
    "splits_250_750": {
        "early_start": 0,
        "early_end":   250,
        "mid_start":   250,
        "mid_end":     750,
        "late_start":  750,
        "late_end":    1000,
    },
    "splits_300_700": {
        "early_start": 0,
        "early_end":   300,
        "mid_start":   300,
        "mid_end":     700,
        "late_start":  700,
        "late_end":    1000,
    },
}

assert BOUNDARY_SET in BOUNDARY_SETS, (
    f"Unknown BOUNDARY_SET='{BOUNDARY_SET}'. "
    f"Valid options: {list(BOUNDARY_SETS.keys())}"
)

bnd = BOUNDARY_SETS[BOUNDARY_SET]

# Canonical CONDITIONS dict â€” t is raw index in [0, 999]
# baseline: cond_fn=None (guidance_scale recorded as 0.0 in CSV)
CONDITIONS = {
    "baseline":       {"t_start": None, "t_end": None},
    "minority-early": {"t_start": bnd["early_start"], "t_end": bnd["early_end"]},
    "minority-mid":   {"t_start": bnd["mid_start"],   "t_end": bnd["mid_end"]},
    "minority-late":  {"t_start": bnd["late_start"],  "t_end": bnd["late_end"]},
    "minority-full":  {"t_start": 0,                  "t_end": 1000},
}

print(f"BOUNDARY_SET={BOUNDARY_SET}")
for cname, cdef in CONDITIONS.items():
    print(f"  {cname}: t_start={cdef['t_start']}, t_end={cdef['t_end']}")

In [ ]:
# â”€â”€ CELL 7: Windowed cond_fn factory (WINDOW-GATING invariant) â”€
#
# When t_start is None (baseline), this factory is not called â€”
# sample_fn receives cond_fn=None instead.
#
# For all other conditions:
#   - If t_val is outside [t_start, t_end), return zeros_like(x).
#   - If t_val is inside the window, compute the minority guidance gradient.
#
# f_extractor is called with f_out=True to return the 8x8x512 bottleneck
# feature map h from EncoderUNetModel.forward, bypassing the classifier head.
# The minority classifier then maps these features to 100 class logits.

def make_cond_fn(t_start, t_end):
    def cond_fn(x, t, y=None):
        assert y is not None
        t_val = t[0].item()
        if not (t_start <= t_val < t_end):
            return th.zeros_like(x)
        with th.enable_grad():
            x_in = x.detach().requires_grad_(True)
            latents = f_extractor(x_in, timesteps=t, f_out=True)
            logits = classifier(latents, timesteps=t)
            log_probs = F.log_softmax(logits, dim=-1)
            selected = log_probs[range(len(logits)), y.view(-1)]
            return th.autograd.grad(selected.sum(), x_in)[0] * args.classifier_scale
    return cond_fn

print("make_cond_fn factory defined.")

In [ ]:
# â”€â”€ CELL 8: Build run list (with GAP_PAIRS continuation support) â”€
#
# Full sweep: 5 conditions x N_SEEDS seeds = 5*N_SEEDS pairs.
# Continuation mode: only the (condition, seed) pairs in GAP_PAIRS are run,
# in the order they appear in GAP_PAIRS.
#
# GAP_PAIRS format: list of [condition_str, seed_int] pairs, e.g.
#   [["minority-mid", 7], ["baseline", 12]]

SEEDS = list(range(1, N_SEEDS + 1))
CONDITION_NAMES = list(CONDITIONS.keys())

if GAP_PAIRS is not None:
    # Continuation mode: run only the specified pairs
    run_list = []
    for pair in GAP_PAIRS:
        condition, seed = pair[0], int(pair[1])
        assert condition in CONDITIONS, (
            f"GAP_PAIRS contains unknown condition '{condition}'. "
            f"Valid: {CONDITION_NAMES}"
        )
        assert 1 <= seed <= N_SEEDS, (
            f"GAP_PAIRS seed {seed} out of range [1, {N_SEEDS}]."
        )
        run_list.append((condition, seed))
    print(f"Continuation mode: {len(run_list)} gap pairs to fill.")
else:
    # Full sweep mode
    run_list = [
        (condition, seed)
        for seed in SEEDS
        for condition in CONDITION_NAMES
    ]
    print(f"Sweep mode: {len(run_list)} runs ({len(CONDITION_NAMES)} conditions x {N_SEEDS} seeds).")

print(f"RUN_MODE={RUN_MODE}, total_planned={len(run_list)}")

In [ ]:
# â”€â”€ CELL 9: CSV setup â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
#
# Canonical column order (all 14 columns always written):
#   run_id, condition, guidance_window, t_start, t_end,
#   guidance_scale, dataset, boundary_set,
#   seed, n_generated, classifier_mean_confidence, classifier_mean_loss,
#   runtime_seconds, artifact_path, timestamp

CSV_FIELDS = [
    "run_id",
    "condition",
    "guidance_window",
    "t_start",
    "t_end",
    "guidance_scale",
    "dataset",
    "boundary_set",
    "seed",
    "n_generated",
    "classifier_mean_confidence",
    "classifier_mean_loss",
    "runtime_seconds",
    "artifact_path",
    "timestamp",
]

scale_str = str(GUIDANCE_SCALE).replace(".", "p")
CSV_PATH = f"/kaggle/working/runs_scale{scale_str}_{DATASET}.csv"

# Write header (overwrites any existing partial CSV for this run)
with open(CSV_PATH, "w", newline="") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=CSV_FIELDS)
    writer.writeheader()

print(f"CSV initialised at: {CSV_PATH}")
print(f"Columns: {CSV_FIELDS}")

In [ ]:
# â”€â”€ CELL 10: Main experiment loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
#
# For each (condition, seed) pair in run_list:
#   1. Build cond_fn (None for baseline, windowed for all others).
#   2. Run p_sample_loop with timestep_respacing=250.
#   3. Evaluate minority classifier on the generated sample.
#   4. Compute classifier_mean_confidence and classifier_mean_loss.
#   5. Write one row to CSV.
#
# Sampler hyperparameters are never modified between conditions.
# Only cond_fn changes.

def model_fn(x, t, y=None):
    """Unconditional model wrapper (class_cond=False)."""
    return model(x, t, None)


def eval_sample(sample_tensor):
    """
    Evaluate the minority classifier on a generated sample.
    Returns (mean_confidence, mean_loss).

    sample_tensor: float32 tensor in [-1, 1], shape (1, 3, 256, 256),
                   as returned by p_sample_loop.

    classifier_mean_confidence: mean of softmax(logits)[y] over n_generated.
    classifier_mean_loss: mean of -log_softmax(logits)[y] over n_generated.
    """
    device = dist_util.dev()
    t_zero = th.zeros(1, dtype=th.long, device=device)
    y_label = th.tensor([MANUAL_CLASS_ID], dtype=th.long, device=device)

    with th.no_grad():
        latents = f_extractor(sample_tensor.to(device), timesteps=t_zero, f_out=True)
        logits = classifier(latents, timesteps=t_zero)
        log_probs = F.log_softmax(logits, dim=-1)
        probs = th.exp(log_probs)

    confidence = probs[0, MANUAL_CLASS_ID].item()
    loss = (-log_probs[0, MANUAL_CLASS_ID]).item()
    return confidence, loss


artifact_dir = f"/kaggle/working/artifacts"
os.makedirs(artifact_dir, exist_ok=True)

completed = 0
errors = []

for condition, seed in run_list:
    cond_def = CONDITIONS[condition]
    t_start = cond_def["t_start"]
    t_end   = cond_def["t_end"]

    # Guidance window string for CSV
    guidance_window = f"({t_start}, {t_end})"

    # Guidance scale: 0.0 for baseline (no guidance applied)
    row_guidance_scale = 0.0 if condition == "baseline" else GUIDANCE_SCALE

    # Build run_id: stable, reproducible, sortable
    cond_short = condition.replace("minority-", "")
    scale_tag  = str(GUIDANCE_SCALE).replace(".", "p")
    bs_tag     = BOUNDARY_SET.replace("_", "-")
    run_id = f"{cond_short}_s{scale_tag}_{bs_tag}_seed{seed:03d}"

    try:
        # Set seed for reproducibility
        th.manual_seed(seed)
        np.random.seed(seed)

        # Build cond_fn
        if condition == "baseline":
            active_cond_fn = None
        else:
            active_cond_fn = make_cond_fn(t_start, t_end)

        model_kwargs = {
            "y": th.tensor([MANUAL_CLASS_ID], device=dist_util.dev(), dtype=th.long)
        }

        t0 = time.time()
        sample = diffusion.p_sample_loop(
            model_fn,
            (1, 3, 256, 256),
            clip_denoised=args.clip_denoised,
            model_kwargs=model_kwargs,
            cond_fn=active_cond_fn,
            device=dist_util.dev(),
        )
        runtime = time.time() - t0

        # Save artifact
        artifact_path = os.path.join(artifact_dir, f"{run_id}.npz")
        sample_uint8 = ((sample + 1) * 127.5).clamp(0, 255).to(th.uint8)
        np.savez(
            artifact_path,
            arr_0=sample_uint8.permute(0, 2, 3, 1).cpu().numpy()
        )

        # Evaluate classifier on the generated sample
        mean_confidence, mean_loss = eval_sample(sample)

        row = {
            "run_id":                    run_id,
            "condition":                 condition,
            "guidance_window":           guidance_window,
            "t_start":                   t_start,
            "t_end":                     t_end,
            "guidance_scale":            row_guidance_scale,
            "dataset":                   DATASET,
            "boundary_set":              BOUNDARY_SET,
            "seed":                      seed,
            "n_generated":               1,
            "classifier_mean_confidence": mean_confidence,
            "classifier_mean_loss":       mean_loss,
            "runtime_seconds":            round(runtime, 2),
            "artifact_path":              artifact_path,
            "timestamp":                  datetime.now(timezone.utc).isoformat(),
        }

        with open(CSV_PATH, "a", newline="") as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=CSV_FIELDS)
            writer.writerow(row)

        completed += 1
        print(
            f"[{completed}/{len(run_list)}] {run_id} "
            f"conf={mean_confidence:.4f} loss={mean_loss:.4f} "
            f"t={runtime:.1f}s"
        )

    except Exception as exc:
        errors.append({"run_id": run_id, "error": str(exc)})
        print(f"ERROR [{run_id}]: {exc}")

print(f"\nDone. {completed}/{len(run_list)} runs completed. {len(errors)} errors.")
if errors:
    print("Failed runs:")
    for e in errors:
        print(f"  {e['run_id']}: {e['error']}")

In [ ]:
# â”€â”€ CELL 11: Display results â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import csv as _csv

print(f"Results CSV: {CSV_PATH}")
print(f"guidance_scale={GUIDANCE_SCALE}, dataset={DATASET}, boundary_set={BOUNDARY_SET}")
print()

if os.path.exists(CSV_PATH):
    with open(CSV_PATH, newline="") as csvfile:
        rows = list(_csv.DictReader(csvfile))

    print(f"Total rows (excluding header): {len(rows)}")
    print()

    # Group by condition and print summary
    from collections import defaultdict
    by_condition = defaultdict(list)
    for row in rows:
        by_condition[row["condition"]].append(row)

    header = f"{'condition':<20} {'seeds':>6} {'mean_conf':>12} {'mean_loss':>12} {'window':>18}"
    print(header)
    print("-" * len(header))
    for cname in ["baseline", "minority-early", "minority-mid", "minority-late", "minority-full"]:
        crows = by_condition.get(cname, [])
        if not crows:
            continue
        confs = [float(r["classifier_mean_confidence"]) for r in crows]
        losses = [float(r["classifier_mean_loss"]) for r in crows]
        window = crows[0]["guidance_window"]
        print(
            f"{cname:<20} {len(crows):>6} "
            f"{sum(confs)/len(confs):>12.4f} "
            f"{sum(losses)/len(losses):>12.4f} "
            f"{window:>18}"
        )
    print()
    print(f"CSV written to: {CSV_PATH}")
else:
    print("CSV not found â€” check for errors above.")